# RF-DETR 1.5 — `AUG_AERIAL`: Augmentations for Overhead Imagery

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roboflow/rf-detr/blob/develop/notebooks/aerial-augmentation-demo.ipynb)

Aerial images break one of detection's core assumptions: objects always
appear upright. From a UAV or satellite, a car travelling north looks
nothing like the same car travelling east. Without rotation augmentation,
a detector trained on mostly north-facing objects will miss the others.

RF-DETR 1.5 ships an **`AUG_AERIAL` preset** built for this problem.
It applies 90° discrete rotations together with horizontal / vertical
flips — giving the model all 8 canonical orientations of every training
image for free.

**What you will learn:**
- Why rotation augmentation is essential for top-down imagery
- How to visually inspect augmented batches *before* training
- How to measure the mAP gain from `AUG_AERIAL` with a side-by-side comparison

## 1. Install RF-DETR 1.5

In [ ]:
!pip install -q rfdetr>=1.5.0 roboflow

## 2. Check GPU

RF-DETR trains on GPU when available and falls back to CPU.
The cell below prints VRAM — a handy sanity check before choosing batch size.

In [ ]:
import os

import torch

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cuda":
    print(f"  GPU  : {torch.cuda.get_device_name(0)}")
    print(f"  VRAM : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

num_workers = min(os.cpu_count() or 2, 8)
print(f"Data-loader workers: {num_workers}")

EPOCHS = 50

## 3. Download an aerial dataset

We use [**Aerial Cows**](https://universe.roboflow.com/roboflow-100/aerial-cows)
from Roboflow 100 — 1,084 drone images of cattle filmed from directly above.
Cows face every compass direction, so rotation augmentation has a real job to do.

The dataset is CC BY 4.0. You need a **free Roboflow API key**:
add it as a Colab secret named `ROBOFLOW_API_KEY` (or set the env var locally).

> To swap in a different aerial dataset, change `WORKSPACE`, `PROJECT`, `VERSION`
> below. Browse https://universe.roboflow.com/browse/aerial for alternatives.

In [ ]:
import os

from rfdetr import RFDETRSmall
from roboflow import Roboflow

WORKSPACE = "roboflow-100"
PROJECT = "aerial-cows"
VERSION = 2

try:
    from google.colab import userdata  # type: ignore[import]
    API_KEY = userdata.get("ROBOFLOW_API_KEY")
except Exception:
    API_KEY = os.environ["ROBOFLOW_API_KEY"]

rf = Roboflow(api_key=API_KEY)
dataset = rf.workspace(WORKSPACE).project(PROJECT).version(VERSION).download("coco")
DATASET_DIR = dataset.location
print(f"Dataset root: {DATASET_DIR}")

## 4. Explore the `AUG_AERIAL` preset

`AUG_AERIAL` is a plain Python dict — inspect, extend, or override it freely.

| Transform | Why it matters for overhead imagery |
|---|---|
| `HorizontalFlip` | Objects can face left or right |
| `VerticalFlip` | Objects can face toward or away from the sensor |
| `Rotate(limit=(90, 90))` | Combined with the flips, covers all 8 cardinal orientations |
| `RandomBrightnessContrast` | Handles lighting changes across time of day, altitude, and cloud cover |

The first three transforms together form the dihedral group D₄ of the
square — meaning the model sees every 90° rotation and reflection of
every training image.

In [ ]:
from rfdetr.datasets.aug_config import AUG_AERIAL

print("AUG_AERIAL preset:")
for transform, params in AUG_AERIAL.items():
    print(f"  {transform}: {params}")

## 5. Train: baseline vs `AUG_AERIAL`

We run two training runs on the same dataset and compare validation mAP@50:95.
The baseline uses no augmentations; the second run uses `AUG_AERIAL`.

> **Tip:** On a T4 GPU with a ~500-image aerial dataset, each 50-epoch run
> takes roughly 5–10 minutes.

In [ ]:
# ── Baseline: no augmentations ──────────────────────────────────────────────
OUTPUT_BASE = "output_baseline"
os.makedirs(OUTPUT_BASE, exist_ok=True)

model_base = RFDETRSmall()
model_base.train(
    dataset_dir=DATASET_DIR,
    epochs=EPOCHS,
    batch_size=8,
    aug_config={},  # no augmentations
    output_dir=OUTPUT_BASE,
    device=device,
    num_workers=num_workers,
    run_test=False,
    progress_bar=True,
)

In [ ]:
# ── With AUG_AERIAL ──────────────────────────────────────────────────────────
OUTPUT_AUG = "output_aerial"
os.makedirs(OUTPUT_AUG, exist_ok=True)

model_aug = RFDETRSmall()
model_aug.train(
    dataset_dir=DATASET_DIR,
    epochs=EPOCHS,
    batch_size=8,
    aug_config=AUG_AERIAL,
    save_dataset_grids=True,  # writes 3×3 grids before training starts
    output_dir=OUTPUT_AUG,
    device=device,
    num_workers=num_workers,
    run_test=False,
    progress_bar=True,
)

### Augmented training batches

RF-DETR writes the grids before the first weight update — you can see exactly
what the model will be trained on.

In [ ]:
from pathlib import Path

import matplotlib.image as mpimg
import matplotlib.pyplot as plt

grids = sorted(Path(OUTPUT_AUG).glob("train_batch*_grid.jpg"))[:3]
if grids:
    fig, axes = plt.subplots(1, len(grids), figsize=(18, 6))
    if len(grids) == 1:
        axes = [axes]
    for ax, g in zip(axes, grids):
        ax.imshow(mpimg.imread(g))
        ax.set_title(g.stem)
        ax.axis("off")
    plt.suptitle("Augmented training batches — AUG_AERIAL", fontsize=13)
    plt.tight_layout()
    plt.show()

## 6. Compare results

RF-DETR writes a `log.txt` (one JSON object per epoch) to `output_dir`.
We parse `test_coco_eval_bbox[0]` — the standard COCO mAP@50:95 — from
both runs and plot them side by side.

In [ ]:
import json

import matplotlib.pyplot as plt


def read_map_history(output_dir: str) -> list[float]:
    """Return per-epoch mAP@50:95 from RF-DETR's log.txt."""
    maps = []
    log_path = Path(output_dir) / "log.txt"
    with open(log_path) as f:
        for line in f:
            entry = json.loads(line.strip())
            bbox = entry.get("test_coco_eval_bbox") or entry.get("ema_test_coco_eval_bbox")
            if bbox:
                maps.append(float(bbox[0]))
    return maps


base_maps = read_map_history(OUTPUT_BASE)
aug_maps = read_map_history(OUTPUT_AUG)

best_base = max(base_maps) if base_maps else float("nan")
best_aug = max(aug_maps) if aug_maps else float("nan")
delta = best_aug - best_base

# ── Side-by-side: training curve  |  best-mAP bar chart ──────────────────────
fig, (ax_curve, ax_bar) = plt.subplots(1, 2, figsize=(14, 4))

# Left: mAP over epochs
ax_curve.plot(base_maps, linewidth=2, label="No augmentation")
ax_curve.plot(aug_maps, linewidth=2, label="AUG_AERIAL")
ax_curve.set_xlabel("Epoch")
ax_curve.set_ylabel("mAP@50:95 (validation)")
ax_curve.set_title("mAP over training")
ax_curve.legend()
ax_curve.grid(alpha=0.3)

# Right: best mAP bar chart
labels = ["No augmentation", "AUG_AERIAL"]
values = [best_base, best_aug]
bars = ax_bar.bar(labels, values, width=0.4)
bars[1].set_color("C1")
ax_bar.set_ylabel("Best mAP@50:95")
ax_bar.set_title(f"Best result  ({delta:+.4f})")
ax_bar.set_ylim(0, max(values) * 1.2)
for bar, val in zip(bars, values):
    ax_bar.text(bar.get_x() + bar.get_width() / 2, val + 0.005, f"{val:.4f}", ha="center")

plt.tight_layout()
plt.show()

print(f"Best mAP@50:95 — no augmentation : {best_base:.4f}")
print(f"Best mAP@50:95 — AUG_AERIAL      : {best_aug:.4f}  ({delta:+.4f})")

## 7. Run inference on a validation image

In [ ]:
import json
from pathlib import Path

import supervision as sv
from PIL import Image

# Load the first image from the validation split
val_ann_path = Path(DATASET_DIR) / "valid" / "_annotations.coco.json"
with open(val_ann_path) as f:
    ann_data = json.load(f)

first_filename = ann_data["images"][0]["file_name"]
image = Image.open(Path(DATASET_DIR) / "valid" / first_filename)

detections = model_aug.predict(image, threshold=0.3)
annotated = sv.BoxAnnotator().annotate(image.copy(), detections)
sv.plot_image(annotated)
print(f"Detections: {len(detections)}")

## Next steps

**Extend the preset** — add transforms that match your specific conditions:
```python
custom_aerial = {
    **AUG_AERIAL,
    "GaussianBlur": {"blur_limit": 3, "p": 0.2},  # UAV motion blur
    "RandomScale":  {"scale_limit": 0.3, "p": 0.4},  # altitude variation
}
```

- [Augmentation docs](https://rfdetr.roboflow.com/develop/learn/train/augmentations/)
- [Advanced training options](https://rfdetr.roboflow.com/develop/learn/train/advanced/)
- [Logger integrations (ClearML, MLflow, W&B)](https://rfdetr.roboflow.com/develop/learn/train/loggers/)
- [Export your model](https://rfdetr.roboflow.com/develop/learn/export/)